# 7장 코드 필사: BERT · BART · ELECTRA

> 파이토치 트랜스포머를 활용한 자연어 처리와 컴퓨터비전 심층학습 — 7장 (pp. 406~451)

## 공통 임포트

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader, TensorDataset, RandomSampler, SequentialSampler

---
## 예제 7.14 GPT-2 문장 분류 모델 평가

> 학습 완료 후 저장된 GPT-2 모델을 불러와 테스트 데이터로 평가

In [2]:
try:
    from transformers import GPT2ForSequenceClassification

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = GPT2ForSequenceClassification.from_pretrained(
        pretrained_model_name_or_path="gpt2",
        num_labels=2,
    ).to(device)
    model.config.pad_token_id = model.config.eos_token_id

    # 저장된 가중치 로드 (학습 완료 후 사용)
    # model.load_state_dict(torch.load("../models/GPT2ForSequenceClassification.pt"))

    # test_loss, test_accuracy = evaluation(model, test_dataloader)
    # print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_accuracy:.4f}")
    print("GPT-2 분류 모델 로드 성공")
    print("예시 출력: Test Loss: 0.4521 | Test Accuracy: 0.8123")

except ImportError as e:
    print(f"[transformers 미설치] pip install transformers")
    print("예시 출력: Test Loss: 0.4521 | Test Accuracy: 0.8123")

[transformers 미설치] pip install transformers
예시 출력: Test Loss: 0.4521 | Test Accuracy: 0.8123


---
## BERT 실습

**데이터셋**: 네이버 영화 리뷰 감성 분석 (NSMC)
**모델**: `bert-base-multilingual-cased`
**태스크**: 이진 감성 분류

### 예제 7.15 네이버 영화 리뷰 데이터 불러오기

> `pip install korpora pandas` 필요

In [3]:
try:
    import pandas as pd
    from Korpora import Korpora

    corpus = Korpora.load("nsmc")
    df = pd.DataFrame({
        "text": corpus.train.texts,
        "label": corpus.train.labels
    }).dropna()

    # 데이터 분할 (학습 60%, 검증 20%, 테스트 20%)
    train, valid, test = np.split(
        df.sample(frac=1, random_state=42),
        [int(0.6 * len(df)), int(0.8 * len(df))]
    )

    print(f"Training Data Size  : {len(train)}")
    print(f"Validation Data Size: {len(valid)}")
    print(f"Testing Data Size   : {len(test)}")
    print("\n샘플:")
    print(train.head(3).to_string(index=False))

except ImportError as e:
    print(f"[Korpora 미설치] pip install korpora pandas")
    print("\n예시 출력:")
    print("Training Data Size  : 12000")
    print("Validation Data Size: 4000")
    print("Testing Data Size   : 4000")
    print("\n샘플:")
    print("징키스칸이란...")

[Korpora 미설치] pip install korpora pandas

예시 출력:
Training Data Size  : 12000
Validation Data Size: 4000
Testing Data Size   : 4000

샘플:
징키스칸이란...


### 예제 7.16 BERT 입력 텐서 생성

In [4]:
try:
    from transformers import BertTokenizer

    epochs = 3
    batch_size = 32
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = BertTokenizer.from_pretrained(
        pretrained_model_name_or_path="bert-base-multilingual-cased",
        do_lower_case=False,  # 대소문자 유지
    )

    def make_dataset(data, tokenizer, device, max_len=128):
        tokenized = tokenizer(
            data["text"].tolist(),
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        input_ids      = tokenized["input_ids"].to(device)
        attention_mask = tokenized["attention_mask"].to(device)
        token_type_ids = tokenized["token_type_ids"].to(device)
        labels = torch.tensor(data["label"].tolist(), dtype=torch.long).to(device)
        return TensorDataset(input_ids, attention_mask, token_type_ids, labels)

    def get_dataloader(dataset, sampler_class, batch_size):
        return DataLoader(dataset, sampler=sampler_class(dataset), batch_size=batch_size)

    print("BertTokenizer 로드 완료")
    print(f"지원 언어: 104개 (bert-base-multilingual-cased)")
    print(f"do_lower_case=False → 대소문자 구분 유지")

    # 데이터셋/로더 생성 (train/valid/test 데이터 필요)
    # train_dataset = make_dataset(train, tokenizer, device)
    # train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)
    # valid_dataset = make_dataset(valid, tokenizer, device)
    # valid_dataloader = get_dataloader(valid_dataset, SequentialSampler, batch_size)
    # test_dataset = make_dataset(test, tokenizer, device)
    # test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

except ImportError:
    print("[transformers 미설치] pip install transformers")

[transformers 미설치] pip install transformers


### BERT 문장 분류 모델 선언 및 구조 확인

In [5]:
try:
    from transformers import BertForSequenceClassification

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model_bert = BertForSequenceClassification.from_pretrained(
        pretrained_model_name_or_path="bert-base-multilingual-cased",
        num_labels=2,
    ).to(device)

    # AdamW: Adam + 가중치 감쇠 (eps: 학습률 0 나누기 방지)
    optimizer_bert = optim.AdamW(model_bert.parameters(), lr=2e-5, eps=1e-8)

    # 모델 구조 출력
    for main_name, main_module in model_bert.named_children():
        print(main_name)
        for sub_name, sub_module in main_module.named_children():
            print(f"  {sub_name}: {sub_module.__class__.__name__}")

except ImportError:
    print("[transformers 미설치]")
    print("\n예상 출력:")
    print("bert")
    print("  embeddings: BertEmbeddings")
    print("    ├── word_embeddings: Embedding")
    print("    ├── position_embeddings: Embedding")
    print("    ├── token_type_embeddings: Embedding")
    print("    └── LayerNorm + Dropout")
    print("  encoder: BertEncoder  (인코더 블록 × 12)")
    print("  pooler: BertPooler    (Linear + Tanh)")
    print("classifier: Linear      (분류 레이어)")

[transformers 미설치]

예상 출력:
bert
  embeddings: BertEmbeddings
    ├── word_embeddings: Embedding
    ├── position_embeddings: Embedding
    ├── token_type_embeddings: Embedding
    └── LayerNorm + Dropout
  encoder: BertEncoder  (인코더 블록 × 12)
  pooler: BertPooler    (Linear + Tanh)
classifier: Linear      (분류 레이어)


### BERT 학습 및 평가 루프

In [6]:
def train_bert(model, optimizer, dataloader):
    model.train()
    train_loss = 0
    for input_ids, attention_mask, token_type_ids, labels in dataloader:
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels,
        )
        loss = outputs.loss
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return train_loss / len(dataloader)

def eval_bert(model, dataloader):
    model.eval()
    val_loss, val_accuracy = 0, 0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for input_ids, attention_mask, token_type_ids, labels in dataloader:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
            logits = outputs.logits
            loss = criterion(logits, labels)
            val_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            val_accuracy += (preds == labels).float().mean().item()
    return val_loss / len(dataloader), val_accuracy / len(dataloader)

print("학습/평가 함수 정의 완료")
print("\n예시 출력:")
print("Epoch 1: Train Loss: 0.4234 | Val Loss: 0.3891 | Val Acc: 0.8023")
print("Epoch 2: Train Loss: 0.2156 | Val Loss: 0.3512 | Val Acc: 0.8512")
print("Epoch 3: Train Loss: 0.1023 | Val Loss: 0.3841 | Val Acc: 0.8085")
print("\nTest Loss: 0.4183 | Test Accuracy: 0.8085")

학습/평가 함수 정의 완료

예시 출력:
Epoch 1: Train Loss: 0.4234 | Val Loss: 0.3891 | Val Acc: 0.8023
Epoch 2: Train Loss: 0.2156 | Val Loss: 0.3512 | Val Acc: 0.8512
Epoch 3: Train Loss: 0.1023 | Val Loss: 0.3841 | Val Acc: 0.8085

Test Loss: 0.4183 | Test Accuracy: 0.8085


---
## BART 실습

**데이터셋**: Argilla 뉴스 요약 데이터셋 (Hugging Face Datasets)
**모델**: `facebook/bart-base`
**태스크**: 뉴스 기사 → 요약문 생성
**평가**: ROUGE 점수

### 예제 7.18 뉴스 요약 데이터셋 불러오기

> `pip install datasets` 필요

In [7]:
try:
    from datasets import load_dataset

    news = load_dataset("argilla/news-summary", split="test")
    df_news = news.to_pandas().sample(5000, random_state=42)[["text", "prediction"]]
    df_news["prediction"] = df_news["prediction"].map(lambda x: x[0]["text"])

    train_news, valid_news, test_news = np.split(
        df_news.sample(frac=1, random_state=42),
        [int(0.6 * len(df_news)), int(0.8 * len(df_news))]
    )

    print(f"Train: {len(train_news)} | Valid: {len(valid_news)} | Test: {len(test_news)}")
    print("\n샘플:")
    print("Text   :", df_news["text"].iloc[0][:80], "...")
    print("Summary:", df_news["prediction"].iloc[0][:80], "...")

except ImportError:
    print("[datasets 미설치] pip install datasets")
    print("\n예시 출력:")
    print("Train: 3000 | Valid: 1000 | Test: 1000")

[datasets 미설치] pip install datasets

예시 출력:
Train: 3000 | Valid: 1000 | Test: 1000


### 예제 7.19 BART 입력 텐서 생성

In [8]:
try:
    from transformers import BartTokenizer

    bart_tokenizer = BartTokenizer.from_pretrained(
        pretrained_model_name_or_path="facebook/bart-base"
    )

    def make_bart_dataset(data, tokenizer, device, max_len=512, summary_max_len=128):
        """본문과 요약문을 토크나이징 → TensorDataset 반환"""
        inputs = tokenizer(
            data["text"].tolist(),
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        with tokenizer.as_target_tokenizer():
            targets = tokenizer(
                data["prediction"].tolist(),
                padding="max_length",
                truncation=True,
                max_length=summary_max_len,
                return_tensors="pt",
            )
        input_ids      = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)
        labels         = targets["input_ids"].to(device)
        # PAD 토큰 위치는 손실 계산에서 제외 (-100)
        labels[labels == tokenizer.pad_token_id] = -100
        return TensorDataset(input_ids, attention_mask, labels)

    print("BartTokenizer 로드 완료: facebook/bart-base")
    print("반환 데이터: (input_ids, attention_mask, labels)")

except ImportError:
    print("[transformers 미설치] pip install transformers")

[transformers 미설치] pip install transformers


### 예제 7.20 BART 모델 선언

In [9]:
try:
    from transformers import BartForConditionalGeneration

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model_bart = BartForConditionalGeneration.from_pretrained(
        pretrained_model_name_or_path="facebook/bart-base"
    ).to(device)

    optimizer_bart = optim.Adam(model_bart.parameters(), lr=5e-5, eps=1e-8)

    # 모델 구조 출력
    for main_name, _ in model_bart.named_children():
        print(main_name)

except ImportError:
    print("[transformers 미설치]")
    print("\n예상 출력:")
    print("shared        ← 인코더/디코더 공유 임베딩")
    print("encoder")
    print("  embed_tokens (shared)")
    print("  embed_positions")
    print("  layers × 6  ← 인코더 블록")
    print("  layernorm_embedding")
    print("decoder")
    print("  embed_tokens (shared)")
    print("  embed_positions")
    print("  layers × 6  ← 디코더 블록")
    print("  layernorm_embedding")

[transformers 미설치]

예상 출력:
shared        ← 인코더/디코더 공유 임베딩
encoder
  embed_tokens (shared)
  embed_positions
  layers × 6  ← 인코더 블록
  layernorm_embedding
decoder
  embed_tokens (shared)
  embed_positions
  layers × 6  ← 디코더 블록
  layernorm_embedding


### ROUGE 평가 지표 계산

> `pip install evaluate rouge_score absl-py` 필요

In [10]:
try:
    import evaluate

    rouge = evaluate.load("rouge")

    # 예시 계산
    predictions = ["대한민국은 8강에 진출하지 못했다"]
    references  = ["대한민국은 16강에 진출했다"]

    result = rouge.compute(predictions=predictions, references=references)
    print("ROUGE 점수 계산 결과:")
    for k, v in result.items():
        print(f"  {k}: {v:.4f}")

except (ImportError, Exception) as e:
    print(f"[평가 라이브러리 미설치] pip install evaluate rouge_score absl-py")
    print("\n예시 ROUGE 점수 결과:")
    print("  rouge1: 0.7143")
    print("  rouge2: 0.3333")
    print("  rougeL: 0.7143")

[평가 라이브러리 미설치] pip install evaluate rouge_score absl-py

예시 ROUGE 점수 결과:
  rouge1: 0.7143
  rouge2: 0.3333
  rougeL: 0.7143


### 예제 7.21 BART 학습 및 평가 루프

In [11]:
def calc_rouge(logits, label_ids, tokenizer, rouge_scorer):
    """로짓 → 텍스트 변환 후 ROUGE 계산"""
    pred_ids = np.argmax(logits, axis=-1)
    # PAD(-100) 위치 복원
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    preds  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    labels = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    result = rouge_scorer.compute(predictions=preds, references=labels)
    return result["rougeL"]

def train_bart(model, optimizer, dataloader):
    model.train()
    train_loss = 0
    for input_ids, attention_mask, labels in dataloader:
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return train_loss / len(dataloader)

def eval_bart(model, dataloader, tokenizer=None, rouge_scorer=None):
    model.eval()
    val_loss, val_rouge = 0, 0
    with torch.no_grad():
        for input_ids, attention_mask, labels in dataloader:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            logits = outputs.logits.detach().cpu().numpy()
            label_ids = labels.cpu().numpy()
            loss = outputs.loss
            val_loss += loss.item()
            if tokenizer and rouge_scorer:
                val_rouge += calc_rouge(logits, label_ids, tokenizer, rouge_scorer)
    return val_loss / len(dataloader), val_rouge / len(dataloader)

print("BART 학습/평가 함수 정의 완료")
print("\n예시 출력:")
print("Epoch 1 | Train Loss: 2.8234 | Val Loss: 2.5123 | Val ROUGE-L: 0.2134")
print("Epoch 2 | Train Loss: 2.1543 | Val Loss: 2.1987 | Val ROUGE-L: 0.2891")
print("Epoch 3 | Train Loss: 1.8234 | Val Loss: 1.9876 | Val ROUGE-L: 0.3127")

BART 학습/평가 함수 정의 완료

예시 출력:
Epoch 1 | Train Loss: 2.8234 | Val Loss: 2.5123 | Val ROUGE-L: 0.2134
Epoch 2 | Train Loss: 2.1543 | Val Loss: 2.1987 | Val ROUGE-L: 0.2891
Epoch 3 | Train Loss: 1.8234 | Val Loss: 1.9876 | Val ROUGE-L: 0.3127


### 예제 7.22 BART 모델 평가 (저장된 모델 로드)

In [12]:
try:
    from transformers import BartForConditionalGeneration

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model_eval = BartForConditionalGeneration.from_pretrained(
        pretrained_model_name_or_path="facebook/bart-base"
    ).to(device)

    # 학습된 가중치 로드
    # model_eval.load_state_dict(
    #     torch.load("../models/BartForConditionalGeneration.pt")
    # )

    # test_loss, test_rouge = eval_bart(model_eval, test_dataloader, bart_tokenizer, rouge)
    # print(f"Test Loss: {test_loss:.4f} | Test ROUGE-L: {test_rouge:.4f}")
    print("BART 평가 모델 로드 완료")
    print("예시: Test Loss: 1.9234 | Test ROUGE-L: 0.3012")

except ImportError:
    print("[transformers 미설치] pip install transformers")
    print("예시: Test Loss: 1.9234 | Test ROUGE-L: 0.3012")

[transformers 미설치] pip install transformers
예시: Test Loss: 1.9234 | Test ROUGE-L: 0.3012


### 예제 7.23 문장 요약문 비교 (Pipeline)

In [13]:
try:
    from transformers import pipeline

    summarizer = pipeline(
        task="summarization",
        model="facebook/bart-large-cnn",  # 요약 특화 사전학습 모델
    )

    # 샘플 뉴스 텍스트 (예시)
    sample_text = (
        "The US space agency NASA has announced plans to return humans to the Moon by 2024 "
        "as part of its Artemis program. The agency says it will land the first woman and "
        "next man on the lunar surface using a combination of commercial and NASA systems. "
        "NASA has been working with multiple companies to develop the spacecraft and landers "
        "needed for the mission."
    )

    result = summarizer(sample_text, max_length=60, min_length=20, do_sample=False)
    print("정답 요약문:", "NASA will return humans to the Moon by 2024.")
    print("모델 요약문:", result[0]["summary_text"])

except ImportError:
    print("[transformers 미설치] pip install transformers")
    print("\n예시 출력:")
    print("정답 요약문: NASA announces return to the Moon by 2024.")
    print("모델 요약문: NASA plans to land the first woman and next man on the Moon as part of the Artemis program.")

[transformers 미설치] pip install transformers

예시 출력:
정답 요약문: NASA announces return to the Moon by 2024.
모델 요약문: NASA plans to land the first woman and next man on the Moon as part of the Artemis program.


---
## ELECTRA 개념 및 구조

> 2020, 구글 — 대체 토큰 탐지(Replaced Token Detection) 기반 사전 학습

In [14]:
# ELECTRA는 개념 설명 중심 — 사전학습 없이 구조만 보여줌

print("=" * 60)
print("BERT MLM vs ELECTRA RTD 비교")
print("=" * 60)
print()
print("[BERT - MLM]")
print("  입력: 나는 [MASK] 갔다")
print("  예측: 학교에")
print("  학습 위치: 마스킹된 15% 위치만")
print()
print("[ELECTRA - RTD]")
print("  원본 문장:  나는 학교에  갔다")
print("  생성기 출력: 나는 [회사에] 갔다  ← 그럴듯한 단어로 대체")
print("  판별기 목표: [O] [X] [O]         ← 각 토큰이 원본인지 탐지")
print("  학습 위치:  모든 토큰 위치 (100%)")
print()
print("[구조]")
print("  Generator  (소형 BERT) → MLM 방식으로 대체 단어 생성")
print("  Discriminator (대형 BERT) → 각 토큰 Original/Replaced 이진 분류")
print("  사전학습 후: Generator 제거, Discriminator만 미세 조정")

BERT MLM vs ELECTRA RTD 비교

[BERT - MLM]
  입력: 나는 [MASK] 갔다
  예측: 학교에
  학습 위치: 마스킹된 15% 위치만

[ELECTRA - RTD]
  원본 문장:  나는 학교에  갔다
  생성기 출력: 나는 [회사에] 갔다  ← 그럴듯한 단어로 대체
  판별기 목표: [O] [X] [O]         ← 각 토큰이 원본인지 탐지
  학습 위치:  모든 토큰 위치 (100%)

[구조]
  Generator  (소형 BERT) → MLM 방식으로 대체 단어 생성
  Discriminator (대형 BERT) → 각 토큰 Original/Replaced 이진 분류
  사전학습 후: Generator 제거, Discriminator만 미세 조정


In [15]:
# 허깅 페이스로 ELECTRA 불러오기 예시 (분류 태스크)
try:
    from transformers import ElectraForSequenceClassification, ElectraTokenizer

    electra_tokenizer = ElectraTokenizer.from_pretrained("google/electra-small-discriminator")
    electra_model = ElectraForSequenceClassification.from_pretrained(
        "google/electra-small-discriminator",
        num_labels=2,
    )
    print("ELECTRA 모델 로드 완료: google/electra-small-discriminator")
    for name, _ in electra_model.named_children():
        print(f"  {name}")

except ImportError:
    print("[transformers 미설치] pip install transformers")
    print("\n예상 구조:")
    print("  electra (BERT와 동일한 인코더 구조)")
    print("  classifier")

[transformers 미설치] pip install transformers

예상 구조:
  electra (BERT와 동일한 인코더 구조)
  classifier
